# O2 A Plot Accessor Demo

This notebook is a catalog of the supported plotting surface: `.plot` accessors on zdisamar domain objects. Each plot is displayed in its own cell.


## Setup

The setup prepares one O2 A case, runs the native diagnostics, and stores one Altair chart per public accessor method.


In [1]:
import zdisamar as zd

O2A_MARKERS_NM = (755.0, 760.76, 776.0)
NOISE_TABLE = (
    [755.0, 760.76, 776.0],
    [900.0, 500.0, 900.0],
)


def spectral_grid(case) -> list[float]:
    start = float(case.spectral_grid.start_nm)
    end = float(case.spectral_grid.end_nm)
    count = int(case.spectral_grid.sample_count)
    if count == 1:
        return [start]
    step = (end - start) / float(count - 1)
    return [start + step * index for index in range(count)]


def nearest_grid_values(grid: list[float], targets_nm: tuple[float, ...]) -> list[float]:
    values = []
    for target in targets_nm:
        nearest = min(grid, key=lambda value: abs(value - target))
        if nearest not in values:
            values.append(nearest)
    return values


def instrument_grid_values(grid: list[float]) -> list[float]:
    values = grid[::35] + nearest_grid_values(grid, O2A_MARKERS_NM)
    return sorted(set(values))

## Build The Charts

This cell has no plot output. It keeps the native objects alive while the accessors materialize their chart data.


In [2]:
case = zd.o2a_disamar_reference_input()
grid = spectral_grid(case)
profile_wavelengths_nm = nearest_grid_values(grid, O2A_MARKERS_NM)
response_wavelengths_nm = instrument_grid_values(grid)

with zd.prepare(case) as prepared:
    with prepared.forward_model(jacobian=True) as spectrum:
        reflectance_chart = spectrum.plot.reflectance()
        radiance_chart = spectrum.plot.radiance()
        irradiance_chart = spectrum.plot.irradiance()
        sun_normalized_radiance_chart = spectrum.plot.sun_normalized_radiance()
        aerosol_jacobian_chart = spectrum.plot.jacobian(state="aerosol_optical_depth")
        snr_chart = spectrum.plot.snr(NOISE_TABLE)
        noise_envelope_chart = spectrum.plot.noise_envelope(NOISE_TABLE)

    with prepared.atmosphere.budget(wavelengths_nm=grid) as budget:
        optical_depth_chart = budget.plot.optical_depth()

    with prepared.collision_induced_absorption.diagnostics(wavelengths_nm=grid) as cia:
        cia_optical_depth_chart = cia.plot.optical_depth()

    with prepared.instrument_response.sampling_table(
        wavelengths_nm=response_wavelengths_nm
    ) as response:
        isrf_chart = response.plot.curve()

summary = {
    "profile_wavelengths_nm": profile_wavelengths_nm,
    "response_wavelength_count": len(response_wavelengths_nm),
    "noise_table": {
        "snr_wavelengths_nm": NOISE_TABLE[0],
        "snr_values": NOISE_TABLE[1],
    },
}
summary

{'profile_wavelengths_nm': [755.0, 760.76, 776.0],
 'response_wavelength_count': 22,
 'noise_table': {'snr_wavelengths_nm': [755.0, 760.76, 776.0],
  'snr_values': [900.0, 500.0, 900.0]}}

## Reflectance


In [3]:
reflectance_chart

alt.LayerChart(...)

## Radiance


In [4]:
radiance_chart

alt.LayerChart(...)

## Irradiance


In [5]:
irradiance_chart

alt.LayerChart(...)

## Sun-Normalized Radiance


In [6]:
sun_normalized_radiance_chart

alt.LayerChart(...)

## Reflectance Jacobian


In [7]:
aerosol_jacobian_chart

alt.LayerChart(...)

## Signal-To-Noise Ratio


In [8]:
snr_chart

alt.LayerChart(...)

## Noise Envelope


In [9]:
noise_envelope_chart

alt.LayerChart(...)

## Atmospheric Optical Depth


In [10]:
optical_depth_chart

alt.Chart(...)

## O2-O2 CIA Optical Depth


In [11]:
cia_optical_depth_chart

alt.Chart(...)

## ISRF Curve


In [12]:
isrf_chart

alt.Chart(...)